# 🚀 BicDRL - Colab 실행 노트북

**Drive 구조:**
```
내 드라이브/26-Spring-Senior-Team2-main/
├── src/, configs/, train.py, evaluate.py ...
└── data/raw/
    ├── No Finding/  (100장)
    └── Nodule/      (10장)
```

In [ ]:
# Step 1: Drive 마운트 & 경로 설정
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/26-Spring-Senior-Team2-main'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print('📂 현재 경로:', os.getcwd())
print('📄 파일:', os.listdir('.'))
print()

# 데이터 확인
for cls in ['No Finding', 'Nodule']:
    d = os.path.join('data', 'raw', cls)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    print(f'  data/raw/{cls}: {n}장')

In [ ]:
# Step 2: 의존성 설치
!pip install -q monai PyYAML

In [ ]:
# Step 3: GPU 확인
!nvidia-smi

In [ ]:
# Step 4: 학습 실행
!python train.py

In [ ]:
# Step 5: 평가
!python evaluate.py

---
## (선택) 노트북 내에서 학습 + 그래프

In [ ]:
import torch, os
from src.utils import get_config, _set_seed
from src.dataset import load_and_filter_data, get_dataloader
from src.environment import MedicalImageEnv
from src.agent import DDQNAgent
from src.replay_buffer import ReplayBuffer

_set_seed(42)
config = get_config('configs/config.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

data_dicts, class_weights = load_and_filter_data(config['data']['csv_path'], config['data']['image_dir'])
train_loader = get_dataloader(data_dicts, batch_size=1)
env = MedicalImageEnv(train_loader, class_weights)

agent = DDQNAgent(num_classes=config['agent']['num_classes'], config=config)
replay_buffer = ReplayBuffer(config['agent']['buffer_capacity'])

num_episodes = config['train']['num_episodes']
batch_size = config['agent']['batch_size']
target_update_freq = config['agent']['target_update_freq']
steps_per_episode = config['train']['steps_per_episode']
save_dir = config['train']['save_dir']
os.makedirs(save_dir, exist_ok=True)

global_step = 0
history = {'episode': [], 'reward': [], 'loss': [], 'epsilon': []}

for episode in range(num_episodes):
    state = env.reset()
    episode_reward = 0
    loss_history = []
    from tqdm import tqdm
    pbar = tqdm(range(steps_per_episode), desc=f'Episode {episode+1}/{num_episodes}')
    for step in pbar:
        action = agent.select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_reward += reward
        replay_buffer.push(state, action, reward, next_state, done)
        state = next_state
        if len(replay_buffer) > batch_size:
            states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
            loss = agent.train_step(states, actions, rewards, next_states, dones)
            loss_history.append(loss)
            agent.decay_epsilon()
        if global_step % target_update_freq == 0:
            agent.update_target_network()
        global_step += 1
    avg_loss = sum(loss_history) / len(loss_history) if loss_history else 0
    history['episode'].append(episode + 1)
    history['reward'].append(episode_reward)
    history['loss'].append(avg_loss)
    history['epsilon'].append(agent.epsilon)
    print(f'Episode {episode+1}/{num_episodes} | Reward: {episode_reward:.2f} | Loss: {avg_loss:.4f} | Eps: {agent.epsilon:.3f}')

torch.save(agent.main_net.state_dict(), os.path.join(save_dir, 'best_model.pth'))
print('🎉 완료!')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['episode'], history['reward'], 'b-o', markersize=4)
axes[0].set_title('Episode Reward')
axes[0].set_xlabel('Episode')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['episode'], history['loss'], 'r-o', markersize=4)
axes[1].set_title('Average Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['episode'], history['epsilon'], 'g-o', markersize=4)
axes[2].set_title('Epsilon Decay')
axes[2].set_xlabel('Episode')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()